# Hardware Trojan Generation and Detection using LLM

- The objective of this Python notebook is to showcase a tool for hardware trojan generation and detection using Large Language Models (LLM)

- Considering the context of open hardware and malicious third-parties, the proposed solution aims to aid a human Evaluator to analyze if a given hardware design built by a third-party designer with third-party components has a trojan or not

- It is necessary to have a Google Colab account and to have a HuggingFace account (PRO account is required to access some models)

## 1. Hardware Trojan Generation

This section presents the automatic hardware trojan generation process in given hardware designs using an open-source LLM from the HuggingFace Cloud.

### 1.1. Obtain Golden Model and setup HF Token

There are some golden models of PQC (Post-Quantum Cryptography) available publicly in GitHub:

- SABER: https://github.com/sujoyetc/SABER_HW/tree/master

- DILITHIUM: https://github.com/GMUCERG/Dilithium/tree/master

- SPHINCS+: https://github.com/slh-dsa/sloth/tree/main

- KYBER: https://github.com/acmert/kyber-polmul-hw/tree/master

In [ ]:
design = """/*
 * From our research paper "High-Performance Hardware Implementation of CRYSTALS-Dilithium"
 * by Luke Beckwith, Duc Tri Nguyen, Kris Gaj
 * at George Mason University, USA
 * https://eprint.iacr.org/2021/1451.pdf
 * =============================================================================
 * Copyright (c) 2021 by Cryptographic Engineering Research Group (CERG)
 * ECE Department, George Mason University
 * Fairfax, VA, U.S.A.
 * Author: Luke Beckwith
 * Licensed under the Apache License, Version 2.0 (the "License");
 * you may not use this file except in compliance with the License.
 * You may obtain a copy of the License at
 *     http://www.apache.org/licenses/LICENSE-2.0
 * Unless required by applicable law or agreed to in writing, software
 * distributed under the License is distributed on an "AS IS" BASIS,
 * WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 * See the License for the specific language governing permissions and
 * limitations under the License.
 * =============================================================================
 * @author   Luke Beckwith <lbeckwit@gmu.edu>
 */


`timescale 1ns / 1ps

module butterfly(
    input             clk,
    input             rst,
    input [2:0]       mode,
    input             validi,
    input signed[23:0]  aj,
    input signed [23:0] ajlen,
    input [23:0]      zeta,
    output reg [23:0] bj,
    output reg [23:0] bjlen,
    output reg        valido
    );

    localparam
        DILITHIUM_Q = 23'd8380417;

    localparam
        FORWARD_NTT_MODE = 3'd0,
        INVERSE_NTT_MODE = 3'd1,
        MULT_MODE        = 3'd2,
        ADD_MODE         = 3'd3,
        SUB_MODE         = 3'd4;


    reg [23:0] aj1 = 0, ajlen1 = 0;
    reg [23:0] aj2 = 0, ajlen2_FNTT = 0;
    reg [23:0] ajlen2_INTT = 0;
    reg [23:0] ajlen2_MODM = 0;
    reg [23:0] aj3  [4:0];
    reg [45:0] mult_result;
    reg [23:0] ajlen3 = 0;
    reg [23:0] aj4 = 0, ajlen4 = 0, ajlen4_sub = 0;
    reg [23:0] aj5 = 0, ajlen5 = 0;


    reg [23:0] adda = 0, addb = 0, suba = 0, subb = 0;
    reg [23:0] adder = 0;
    reg [24:0] add_tmp = 0;
    reg [23:0] sub_tmp = 0;
    reg [23:0] subtractor = 0;

    reg [9:0] valid_sr;

    wire barrett_readyi;
    reg  barrett_validi;
    (*keep="true"*)reg  [45:0] barrett_datai;
    wire barrett_valido;
    reg  barrett_readyo;
    wire [22:0] barrett_remainder;
    wire [23:0] barrett_quotient;

    reg [23:0] multa, multb;
    reg [23:0] zeta_delay,zeta_delay2;

    Barrett REDUCER(
      clk,
      rst,
      barrett_readyi,
      barrett_validi,
      barrett_datai,
      barrett_readyo,
      barrett_valido,
      barrett_remainder,
      barrett_quotient
    );

    initial begin
        valid_sr = 0;
        aj3[0] = 0;
        aj3[1] = 0;
        aj3[2] = 0;
        aj3[3] = 0;
        aj3[4] = 0;
    end

    reg [2:0] modei;

    always @(*) begin
        adda   = 0;
        addb   = 0;
        suba   = 0;
        subb   = 0;
        valido = 0;

        barrett_readyo = 1;

        multa = ajlen2_FNTT;
        multb = (modei == INVERSE_NTT_MODE) ? zeta_delay2 : zeta_delay;

        if (mode == INVERSE_NTT_MODE)
            valido = valid_sr[8];
        else if (mode == MULT_MODE)
            valido = valid_sr[7];
        else if (mode == FORWARD_NTT_MODE)
            valido = valid_sr[7];
        else
            valido = valid_sr[3];

        case(modei)
        FORWARD_NTT_MODE: begin
            adda = aj3[4];
            addb = ajlen3;

            suba = aj3[4];
            subb = ajlen3;

            multa = ajlen2_FNTT;
        end
        INVERSE_NTT_MODE: begin
            multa = ajlen2_INTT;

            adda = aj1;
            addb = ajlen1;

            suba = aj1;
            subb = ajlen1;
        end
        MULT_MODE: begin
            // MULT-ACC
            multa = ajlen2_MODM;

            adda = aj4;
            addb = ajlen3;
        end
        ADD_MODE: begin
            adda = ajlen1;
            addb = zeta_delay;

            suba = ajlen4;
            subb = DILITHIUM_Q;
        end
        SUB_MODE: begin
            adda = ajlen1;
            addb = DILITHIUM_Q;

            suba = aj4;
            subb = ajlen4_sub;
        end
        endcase

        barrett_validi = 1;

        adder       = adda + addb;
        mult_result = multb * ajlen2_INTT;
        sub_tmp     = suba + DILITHIUM_Q;
        subtractor  = (subb > suba) ? sub_tmp - subb : suba - subb;
    end



    always @(posedge clk) begin
        if (rst) begin
            valid_sr <= 0;
        end else begin
            valid_sr <= {valid_sr[8:0], validi};
        end

        modei <= mode;

        zeta_delay <= (mode == INVERSE_NTT_MODE) ? DILITHIUM_Q - zeta :  zeta;
        zeta_delay2 <= zeta_delay;

        aj1    <= aj;
        ajlen1 <= ajlen;

        barrett_datai  <= mult_result;

        bj    <= (aj5 >= DILITHIUM_Q)    ? aj5    - DILITHIUM_Q : aj5;
        bjlen <= (ajlen5 >= DILITHIUM_Q) ? ajlen5 - DILITHIUM_Q : ajlen5;

        ajlen3 <= barrett_remainder;

        ajlen4 <= adder;
        ajlen4_sub <= zeta_delay;
        case(mode)
        FORWARD_NTT_MODE: begin
            aj2    <= aj;
            ajlen2_INTT <= ajlen;

            ajlen5 <= subtractor;
            aj5    <= adder;

        end
        INVERSE_NTT_MODE: begin
            aj2    <= adder;
            ajlen2_INTT <= subtractor;

            if (aj3[4][0] == 1)
                aj5 <= (aj3[4] >> 1) + (DILITHIUM_Q + 1) / 2;
            else
                aj5 <= (aj3[4] >> 1);

            if (ajlen3[0] == 1)
                ajlen5 <= (ajlen3 >> 1) + (DILITHIUM_Q + 1) / 2;
            else
                ajlen5 <= (ajlen3 >> 1);
        end
        MULT_MODE: begin
            aj2    <= aj;
            aj4    <= aj3[3];

            ajlen2_INTT <= ajlen;
            ajlen5 <= adder;
        end
        ADD_MODE: begin
            ajlen5 <= (ajlen4 > DILITHIUM_Q) ? subtractor : ajlen4;
        end
        SUB_MODE: begin
            aj4    <= (ajlen1 < zeta_delay) ? adder : ajlen1;
            ajlen5 <= subtractor;
        end
        endcase


        aj3[0] <= aj2;
        aj3[1] <= aj3[0];
        aj3[2] <= aj3[1];
        aj3[3] <= aj3[2];
        aj3[4] <= aj3[3];
    end



endmodule"""

In [ ]:
design

'/*\n * From our research paper "High-Performance Hardware Implementation of CRYSTALS-Dilithium"\n * by Luke Beckwith, Duc Tri Nguyen, Kris Gaj\n * at George Mason University, USA\n * https://eprint.iacr.org/2021/1451.pdf\n * =============================================================================\n * Copyright (c) 2021 by Cryptographic Engineering Research Group (CERG)\n * ECE Department, George Mason University\n * Fairfax, VA, U.S.A.\n * Author: Luke Beckwith\n * Licensed under the Apache License, Version 2.0 (the "License");\n * you may not use this file except in compliance with the License.\n * You may obtain a copy of the License at\n *     http://www.apache.org/licenses/LICENSE-2.0\n * Unless required by applicable law or agreed to in writing, software\n * distributed under the License is distributed on an "AS IS" BASIS,\n * WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n * See the License for the specific language governing permissions and\n * l

In [ ]:
from google.colab import userdata
token = userdata.get('HF_TOKEN')

### 1.2. Remove Comments

In [ ]:
import re, os

def remove_comments(design):

    # Regular expressions to match single-line and multi-line comments
    pattern = r"(--.*?$|/\*.*?\*/)"

    # Remove comments using regex
    design_no_comments = re.sub(pattern, "", design, flags=re.MULTILINE | re.DOTALL)

    # Remove extra newline characters
    design_no_comments = os.linesep.join([s for s in design_no_comments.splitlines() if s])

    return design_no_comments

In [ ]:
design_no_comments = remove_comments(design)
design_no_comments

'`timescale 1ns / 1ps\nmodule butterfly(\n    input             clk,\n    input             rst,\n    input [2:0]       mode,\n    input             validi,\n    input signed[23:0]  aj,\n    input signed [23:0] ajlen,\n    input [23:0]      zeta,\n    output reg [23:0] bj,\n    output reg [23:0] bjlen,\n    output reg        valido\n    );\n    localparam\n        DILITHIUM_Q = 23\'d8380417;\n    localparam\n        FORWARD_NTT_MODE = 3\'d0,\n        INVERSE_NTT_MODE = 3\'d1,\n        MULT_MODE        = 3\'d2,\n        ADD_MODE         = 3\'d3,\n        SUB_MODE         = 3\'d4;\n    reg [23:0] aj1 = 0, ajlen1 = 0;\n    reg [23:0] aj2 = 0, ajlen2_FNTT = 0;\n    reg [23:0] ajlen2_INTT = 0;\n    reg [23:0] ajlen2_MODM = 0;\n    reg [23:0] aj3  [4:0];\n    reg [45:0] mult_result;\n    reg [23:0] ajlen3 = 0;\n    reg [23:0] aj4 = 0, ajlen4 = 0, ajlen4_sub = 0;\n    reg [23:0] aj5 = 0, ajlen5 = 0;\n    reg [23:0] adda = 0, addb = 0, suba = 0, subb = 0;\n    reg [23:0] adder = 0;\n    reg [2

### 1.3. Analyze Golden Model

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def analyze_design(verilog, model_id):
  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  try:

    messages = [
      { "role": "system", "content": """You are an expert in hardware development and design.""" },
      { "role": "user", "content": "Given the following Verilog design, explain it: " + verilog }
    ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']

    return response_content

  except Exception as e:
    print(f"An error occurred: {e}")
    return 0

In [ ]:
analyze_design(design, "meta-llama/Llama-3.3-70B-Instruct")

'This Verilog module implements a butterfly operation, which is a key component in the Number Theoretic Transform (NTT) algorithm used in the CRYSTALS-Dilithium cryptographic scheme. The module takes several inputs, including:\n\n* `clk`: the clock signal\n* `rst`: the reset signal\n* `mode`: a 3-bit signal that determines the operation mode (forward NTT, inverse NTT, multiplication, addition, or subtraction)\n* `validi`: a signal that indicates whether the input data is valid\n* `aj` and `ajlen`: 24-bit signed input data\n* `zeta`: a 24-bit input data\n\nThe module produces several outputs, including:\n\n* `bj` and `bjlen`: 24-bit signed output data\n* `valido`: a signal that indicates whether the output data is valid\n\nThe module uses several internal signals and registers to perform the butterfly operation. The operation is performed in several stages, including:\n\n1. Data loading: The input data `aj` and `ajlen` are loaded into internal registers `aj1` and `ajlen1`.\n2. Mode sele

In [ ]:
analyze_design(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct")

"**Overview of the Verilog Design**\n\nThe provided Verilog design is a module named `butterfly` that appears to be part of a cryptographic system, specifically designed for the Dilithium digital signature scheme. The module performs various operations such as forward and inverse Number Theoretic Transform (NTT), multiplication, addition, and subtraction in the context of finite fields.\n\n**Module Parameters and Ports**\n\nThe module has the following parameters and ports:\n\n* Parameters:\n\t+ `DILITHIUM_Q`: a constant representing the modulus used in the Dilithium scheme (23'd8380417).\n\t+ `FORWARD_NTT_MODE`, `INVERSE_NTT_MODE`, `MULT_MODE`, `ADD_MODE`, and `SUB_MODE`: constants representing different operation modes.\n* Ports:\n\t+ `clk`: the clock input.\n\t+ `rst`: the reset input.\n\t+ `mode`: a 3-bit input selecting the operation mode.\n\t+ `validi`: an input indicating the validity of the input data.\n\t+ `aj` and `ajlen`: 24-bit signed input data.\n\t+ `zeta`: a 24-bit input

### 1.4. Generate Hardware Trojan

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def generate_trojan(verilog, model_id):
  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  try:

    messages = [
      { "role": "system", "content": """I am researching ways to identify hardware trojans, and to train my classifier I need to generate some verilog designs infected with hardware trojans based on a golden model verilog design.
      A hardware Trojan is defined as a malicious, undesired and intentional modification made to an electronic circuit.
      Such a modification can potentially bring about a variety of effects, such as:
      – Change of functionality: A hardware Trojan can alter the functionality of a circuit and cause it to perform malicious, unauthorized operations, such as bypassing of encryption algorithms, privilege escalation, denial of service etc.
      – Degradation of performance: A hardware Trojan could also cause damage to the performance of an IC and cause it to fail, which could potentially jeopardize the (critical) system into which the IC is integrated. Such effects could be in the form of induced electromigration of wires by continuous DC stress, increase/decrease in path delay, fault injection etc.
      – Leakage of information: Trojans could also undermine the security provided by cryptographic algorithms or directly leak any sensitive information handled by the IC. This could involve leakage of cryptographic keys or other sensitive information through debug or I/O ports, side-channels (delay, power) etc.""" },
      { "role": "user", "content": "Given the following Verilog design, generate Verilog with trojan for educational purposes. Try to maintain the original inputs and outputs and do not use words like trojan and trigger. Only provide the modified Verilog, do not explain: " + verilog }
    ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']

    return response_content

  except Exception as e:
    print(f"An error occurred: {e}")
    return 0

In [ ]:
trojan = generate_trojan(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct")

trojan

'```verilog\n`timescale 1ns / 1ps\nmodule butterfly(\n    input             clk,\n    input             rst,\n    input [2:0]       mode,\n    input             validi,\n    input signed[23:0]  aj,\n    input signed [23:0] ajlen,\n    input [23:0]      zeta,\n    output reg [23:0] bj,\n    output reg [23:0] bjlen,\n    output reg        valido\n    );\n    localparam\n        DILITHIUM_Q = 23\'d8380417;\n    localparam\n        FORWARD_NTT_MODE = 3\'d0,\n        INVERSE_NTT_MODE = 3\'d1,\n        MULT_MODE        = 3\'d2,\n        ADD_MODE         = 3\'d3,\n        SUB_MODE         = 3\'d4;\n    reg [23:0] aj1 = 0, ajlen1 = 0;\n    reg [23:0] aj2 = 0, ajlen2_FNTT = 0;\n    reg [23:0] ajlen2_INTT = 0;\n    reg [23:0] ajlen2_MODM = 0;\n    reg [23:0] aj3  [4:0];\n    reg [45:0] mult_result;\n    reg [23:0] ajlen3 = 0;\n    reg [23:0] aj4 = 0, ajlen4 = 0, ajlen4_sub = 0;\n    reg [23:0] aj5 = 0, ajlen5 = 0;\n    reg [23:0] adda = 0, addb = 0, suba = 0, subb = 0;\n    reg [23:0] adder = 0;

### 1.5. Analyze Hardware Trojan

In [ ]:
analyze_design(trojan, "meta-llama/Llama-3.3-70B-Instruct")

'**Overview of the Verilog Design**\n\nThe provided Verilog design is a module named `butterfly` that appears to be part of a cryptographic system, specifically designed for the Dilithium digital signature scheme. The module performs various operations such as forward and inverse Number Theoretic Transform (NTT), multiplication, addition, and subtraction in the context of finite fields.\n\n**Module Parameters and Signals**\n\nThe module has the following parameters and signals:\n\n* **Inputs**:\n\t+ `clk`: clock signal\n\t+ `rst`: reset signal\n\t+ `mode`: 3-bit signal indicating the operation mode (forward NTT, inverse NTT, multiplication, addition, or subtraction)\n\t+ `validi`: input validity signal\n\t+ `aj` and `ajlen`: 24-bit signed input values\n\t+ `zeta`: 24-bit input value\n* **Outputs**:\n\t+ `bj` and `bjlen`: 24-bit output values\n\t+ `valido`: output validity signal\n\n**Internal Signals and Registers**\n\nThe module uses several internal signals and registers to perform t

## 2. Hardware Trojan Detection

This section presents the hardware trojan detection using LLM based on role-playing.

#### 2.1. Simple Prompt

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_simple_prompt(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
      { "role": "system", "content": "You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs." },
      { "role": "user", "content": "Evaluate if the following Verilog design has a trojan or not, and justify: " + verilog }
      ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
        { "role": "system", "content": "You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs. Given a Verilog design, you can only answer with 1 if it has trojan or 0 otherwise. Do not deviate from the 0 or 1 answer pattern." },
        { "role": "user", "content": "Evaluate if the following Verilog design has a trojan or not, only return numerical value based on your analysis: " + verilog }
        ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_simple_prompt(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_simple_prompt(design, "meta-llama/Llama-3.3-70B-Instruct", True)

'To evaluate if the given Verilog design has a trojan or not, we need to analyze the code for any suspicious or malicious behavior. Here are some observations and justifications:\n\n1. **Complexity and Obfuscation**: The code appears to be a part of a cryptographic implementation, specifically a butterfly module for the CRYSTALS-Dilithium algorithm. The complexity of the code and the use of various signals, registers, and modules make it challenging to analyze. However, this complexity alone does not necessarily indicate a trojan.\n\n2. **Unconventional Signal Names and Usage**: Some signal names, such as `aj3`, `ajlen2_FNTT`, `ajlen2_INTT`, and `ajlen2_MODM`, seem unconventional and might be intended to obscure the code\'s functionality. Nevertheless, without further context or evidence, it\'s difficult to conclude that these names are malicious.\n\n3. **Potential Backdoor**: The `valid_sr` signal is a 10-bit shift register that seems to be used to delay the `validi` signal. However, 

In [ ]:
detect_trojan_simple_prompt(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_simple_prompt(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

"To evaluate if the given Verilog design has a hardware trojan or not, we need to analyze the code for any suspicious or malicious behavior. Here are some potential red flags:\n\n1. **Uninitialized variables**: The code has several uninitialized variables, such as `aj3`, `ajlen2_FNTT`, `ajlen2_INTT`, and `ajlen2_MODM`. Although some of these variables are initialized in the `initial` block, it's still possible that they may not be properly initialized in all scenarios.\n2. **Unused signals**: The code has several unused signals, such as `barrett_readyi`, `barrett_valido`, and `barrett_quotient`. While these signals may not be used in the current implementation, they could potentially be used to inject malicious behavior.\n3. **Complex conditional statements**: The code has several complex conditional statements, such as the `case` statement in the `always @(*)` block. These statements can make it difficult to understand the intended behavior of the code and may provide opportunities fo

In [ ]:
detect_trojan_simple_prompt(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_simple_prompt(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

"Evaluating the given Verilog design for potential hardware trojans requires a thorough analysis of the code. Here are some observations and potential red flags:\n\n1. **Unconventional use of variables**: The design uses a mix of synchronous and combinational logic, which can make it harder to analyze. For example, the `valid_sr` register is used as a shift register, but its purpose is not immediately clear.\n2. **Unused or redundant code**: There are several instances of unused or redundant code, such as the `aj3` array, which is initialized but not used consistently throughout the design. This could be a sign of a trojan or a mistake.\n3. **Complex conditional statements**: The design features complex conditional statements, such as the `case` statement inside the `always @(*)` block. While this is not necessarily a red flag, it can make the code harder to analyze and increase the risk of errors or trojans.\n4. **Potential for data manipulation**: The design performs various arithmet

#### 2.2. TrustHub Benchmark Information

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_benchmark_definition(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
    { "role": "system", "content": """You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs.
    A hardware Trojan is defined as a malicious, undesired and intentional modification made to an electronic circuit.
    Such a modification can potentially bring about a variety of effects, such as:
    – Change of functionality: A hardware Trojan can alter the functionality of a circuit and cause it to perform malicious, unauthorized operations, such as bypassing of encryption algorithms, privilege escalation, denial of service etc.
    – Degradation of performance: A hardware Trojan could also cause damage to the performance of an IC and cause it to fail, which could potentially jeopardize the (critical) system into which the IC is integrated. Such effects could be in the form of induced electromigration of wires by continuous DC stress, increase/decrease in path delay, fault injection etc.
    – Leakage of information: Trojans could also undermine the security provided by cryptographic algorithms or directly leak any sensitive information handled by the IC. This could involve leakage of cryptographic keys or other sensitive information through debug or I/O ports, side-channels (delay, power) etc.
    For example, consider this hardware trojan example found in TrustHub (AES-T2300):
    module aes_128(clk, rst, state, key, out);
    input          clk;
    input          rst;
    input  [127:0] state, key;
    output [127:0] out;
    wire   [127:0] out2;
    reg    [127:0] s0, k0;
    wire   [127:0] s1, s2, s3, s4, s5, s6, s7, s8, s9,
                   k1, k2, k3, k4, k5, k6, k7, k8, k9,
                   k0b, k1b, k2b, k3b, k4b, k5b, k6b, k7b, k8b, k9b;
    wire Tj_Trig;
    always @ (posedge clk)
      begin
        s0 <= state ^ key;
        k0 <= key;
      end
    expand_key_128
        a1 (clk, k0, k1, k0b, 8'h1),
        a2 (clk, k1, k2, k1b, 8'h2),
        a3 (clk, k2, k3, k2b, 8'h4),
        a4 (clk, k3, k4, k3b, 8'h8),
        a5 (clk, k4, k5, k4b, 8'h10),
        a6 (clk, k5, k6, k5b, 8'h20),
        a7 (clk, k6, k7, k6b, 8'h40),
        a8 (clk, k7, k8, k7b, 8'h80),
        a9 (clk, k8, k9, k8b, 8'h1b),
       a10 (clk, k9,   , k9b, 8'h36);
    one_round
        r1 (clk, s0, k0b, s1),
        r2 (clk, s1, k1b, s2),
        r3 (clk, s2, k2b, s3),
        r4 (clk, s3, k3b, s4),
        r5 (clk, s4, k4b, s5),
        r6 (clk, s5, k5b, s6),
        r7 (clk, s6, k6b, s7),
        r8 (clk, s7, k7b, s8),
        r9 (clk, s8, k8b, s9);
    final_round
        rf (clk, s9, k9b, out2);
    TSC Trojan(s2[89], s5[121], Tj_Trig);
    assign out[127:1] = out2[127:1];
    assign out[0] = out2[0] ^ Tj_Trig;
    endmodule
    module TSC(
        input r1,
        input r2,
      output trigger
        );
      assign trigger = r1 && r2;
      always @ (posedge trigger) 	$display("Trojan Activated");
    endmodule""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not, and justify: " + verilog}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
    { "role": "system", "content": """You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs.
    A hardware Trojan is defined as a malicious, undesired and intentional modification made to an electronic circuit.
    Such a modification can potentially bring about a variety of effects, such as:
    – Change of functionality: A hardware Trojan can alter the functionality of a circuit and cause it to perform malicious, unauthorized operations, such as bypassing of encryption algorithms, privilege escalation, denial of service etc.
    – Degradation of performance: A hardware Trojan could also cause damage to the performance of an IC and cause it to fail, which could potentially jeopardize the (critical) system into which the IC is integrated. Such effects could be in the form of induced electromigration of wires by continuous DC stress, increase/decrease in path delay, fault injection etc.
    – Leakage of information: Trojans could also undermine the security provided by cryptographic algorithms or directly leak any sensitive information handled by the IC. This could involve leakage of cryptographic keys or other sensitive information through debug or I/O ports, side-channels (delay, power) etc.
    For example, consider this hardware trojan example found in TrustHub (AES-T2300):
    module aes_128(clk, rst, state, key, out);
    input          clk;
    input          rst;
    input  [127:0] state, key;
    output [127:0] out;
    wire   [127:0] out2;
    reg    [127:0] s0, k0;
    wire   [127:0] s1, s2, s3, s4, s5, s6, s7, s8, s9,
                   k1, k2, k3, k4, k5, k6, k7, k8, k9,
                   k0b, k1b, k2b, k3b, k4b, k5b, k6b, k7b, k8b, k9b;
    wire Tj_Trig;
    always @ (posedge clk)
      begin
        s0 <= state ^ key;
        k0 <= key;
      end
    expand_key_128
        a1 (clk, k0, k1, k0b, 8'h1),
        a2 (clk, k1, k2, k1b, 8'h2),
        a3 (clk, k2, k3, k2b, 8'h4),
        a4 (clk, k3, k4, k3b, 8'h8),
        a5 (clk, k4, k5, k4b, 8'h10),
        a6 (clk, k5, k6, k5b, 8'h20),
        a7 (clk, k6, k7, k6b, 8'h40),
        a8 (clk, k7, k8, k7b, 8'h80),
        a9 (clk, k8, k9, k8b, 8'h1b),
       a10 (clk, k9,   , k9b, 8'h36);
    one_round
        r1 (clk, s0, k0b, s1),
        r2 (clk, s1, k1b, s2),
        r3 (clk, s2, k2b, s3),
        r4 (clk, s3, k3b, s4),
        r5 (clk, s4, k4b, s5),
        r6 (clk, s5, k5b, s6),
        r7 (clk, s6, k6b, s7),
        r8 (clk, s7, k7b, s8),
        r9 (clk, s8, k8b, s9);
    final_round
        rf (clk, s9, k9b, out2);
    TSC Trojan(s2[89], s5[121], Tj_Trig);
    assign out[127:1] = out2[127:1];
    assign out[0] = out2[0] ^ Tj_Trig;
    endmodule
    module TSC(
        input r1,
        input r2,
      output trigger
        );
      assign trigger = r1 && r2;
      always @ (posedge trigger) 	$display("Trojan Activated");
    endmodule""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not: " + verilog + """.
    Don't explain anything just answer with 1 if it probably has trojan or 0 otherwise.
    Do not deviate from the 0 or 1 answer pattern."""}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_benchmark_definition(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_benchmark_definition(design, "meta-llama/Llama-3.3-70B-Instruct", True)

"After analyzing the provided Verilog design, I did not find any obvious signs of a hardware Trojan. Here's why:\n\n1. **No suspicious modules or instances**: The design consists of a single module `butterfly` with a clear and well-structured architecture. There are no unusual or unexplained modules or instances that could be indicative of a Trojan.\n2. **No hidden or unused signals**: All signals in the design seem to be used and connected to other modules or logic. There are no unused or hidden signals that could be used to trigger a Trojan.\n3. **No unusual or unexplained logic**: The design's logic appears to be consistent with the functionality described in the comments and the research paper. There are no unusual or unexplained logic blocks that could be indicative of a Trojan.\n4. **No backdoors or secret inputs**: There are no apparent backdoors or secret inputs that could be used to trigger a Trojan.\n5. **No suspicious timing or clocking**: The design's timing and clocking se

In [ ]:
detect_trojan_benchmark_definition(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_benchmark_definition(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

'To evaluate if the given Verilog design has a trojan or not, we need to analyze the code for any suspicious or malicious behavior. Here are some observations and justifications:\n\n1. **Complexity and Obfuscation**: The code is quite complex and has many conditional statements, which can make it harder to understand and analyze. However, this complexity alone does not necessarily indicate a trojan.\n\n2. **Unusual or Unexplained Behavior**: There are no obvious unusual or unexplained behaviors in the code. The design appears to be implementing a butterfly operation, which is a common component in cryptographic algorithms.\n\n3. **Hidden or Unintended Functionality**: After analyzing the code, there is no clear evidence of hidden or unintended functionality that could be used to compromise the security of the system.\n\n4. **Backdoors or Secret Keys**: There are no apparent backdoors or secret keys embedded in the code.\n\n5. **Anomalous Resource Usage**: The design uses a Barrett redu

In [ ]:
detect_trojan_benchmark_definition(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_benchmark_definition(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

"To evaluate if the given Verilog design has a Trojan or not, we need to analyze the code for any suspicious or malicious behavior. Here are some observations and justifications:\n\n1. **Complexity and Obfuscation**: The code is quite complex and has many conditional statements, which can make it harder to understand and analyze. However, this complexity alone does not necessarily indicate a Trojan.\n\n2. **Unusual or Redundant Code**: There are some unusual or redundant code segments, such as the `aj3` array, which is initialized with zeros but not used extensively. However, this could be a legitimate design choice or a leftover from a previous version of the code.\n\n3. **Potential Backdoors**: There are no obvious backdoors or secret inputs that could be used to trigger malicious behavior. However, the `mode` input and the various conditional statements based on it could potentially be used to create a Trojan.\n\n4. **Information Leakage**: There are no obvious information leakage p

#### 2.3. Checklist-based Review

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_checklist(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
      { "role": "system", "content": """You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs. When analyzing a Verilog description of a circuit to detect hardware Trojans, it is important to consider that hardware Trojans are designed to evade detection and can have a variety of trigger mechanisms and payloads
. Therefore, the analysis should focus on identifying any suspicious logic or behavior that deviates from the expected functionality. Here are key aspects to focus on:
●
Rarely Triggered Logic: Hardware Trojans are often designed to activate only under specific, rare conditions to avoid detection during testing
. This means it's critical to identify any logic in the Verilog description that is only activated by unusual input combinations or sequences. These could be potential Trojan triggers. Analyzing the circuit's state space and controllability can help pinpoint these rare activation conditions
.
●
Unexecuted Code: Some hardware Trojans might reside in sections of the code that are never executed during normal operation
. Analyzing code coverage can help uncover these hidden portions of the code, which could potentially harbor malicious logic. Using line, toggle, statement, and Finite State Machine coverage algorithms can help identify these unexecuted code sections
.
●
Redundant or Unexpected Logic: Hardware Trojans often involve adding extra circuitry or modifying existing circuitry to implement their malicious functionality
. Thoroughly reviewing the Verilog description for any logic that appears redundant, unnecessary, or inconsistent with the circuit's intended functionality is crucial. Such anomalies could be signs of Trojan payloads. Comparing the circuit's structure with a known-good reference, if available, can help identify structural modifications
.
●
Deviations from Specifications: Hardware Trojans can cause the circuit to deviate from its intended behavior, especially under specific conditions or corner cases
. Comparing the circuit's behavior as described in the Verilog code with its formal specifications is essential. Any discrepancies, especially under rare or boundary conditions, could point towards the presence of a Trojan. Formal verification techniques can help prove or disprove that the circuit adheres to its security properties
.
●
Untestable Stuck-at Faults: While not directly related to hardware Trojans, untestable stuck-at faults can complicate Trojan detection
. ATPG techniques can help identify these faults, and removing the corresponding gates from the circuit can simplify the analysis and potentially make Trojan detection easier
.
●
Timing Anomalies: Hardware Trojans might introduce unexpected delays or timing variations in the circuit
. Analyzing the timing characteristics of the circuit as described in the Verilog code, especially under different operating conditions, is crucial. Any significant deviations from the expected timing behavior could indicate the presence of a Trojan. Techniques like path delay fingerprinting can be used to compare timing characteristics with a golden reference
.
●
Suspicious Signals: Certain signals within the circuit might exhibit unusual behavior or patterns that could indicate malicious activity
. These could include signals with unusually low or high switching probabilities, signals that are highly correlated with specific inputs, or signals that appear to be used for unintended purposes. Techniques like signal correlation analysis and clustering can help identify suspicious signals. Equivalence checking can be used to compare these signals against the specifications from the IP provider.""" },
      { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not, and justify: " + verilog}
      ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
      { "role": "system", "content": """You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs. When analyzing a Verilog description of a circuit to detect hardware Trojans, it is important to consider that hardware Trojans are designed to evade detection and can have a variety of trigger mechanisms and payloads
. Therefore, the analysis should focus on identifying any suspicious logic or behavior that deviates from the expected functionality. Here are key aspects to focus on:
●
Rarely Triggered Logic: Hardware Trojans are often designed to activate only under specific, rare conditions to avoid detection during testing
. This means it's critical to identify any logic in the Verilog description that is only activated by unusual input combinations or sequences. These could be potential Trojan triggers. Analyzing the circuit's state space and controllability can help pinpoint these rare activation conditions
.
●
Unexecuted Code: Some hardware Trojans might reside in sections of the code that are never executed during normal operation
. Analyzing code coverage can help uncover these hidden portions of the code, which could potentially harbor malicious logic. Using line, toggle, statement, and Finite State Machine coverage algorithms can help identify these unexecuted code sections
.
●
Redundant or Unexpected Logic: Hardware Trojans often involve adding extra circuitry or modifying existing circuitry to implement their malicious functionality
. Thoroughly reviewing the Verilog description for any logic that appears redundant, unnecessary, or inconsistent with the circuit's intended functionality is crucial. Such anomalies could be signs of Trojan payloads. Comparing the circuit's structure with a known-good reference, if available, can help identify structural modifications
.
●
Deviations from Specifications: Hardware Trojans can cause the circuit to deviate from its intended behavior, especially under specific conditions or corner cases
. Comparing the circuit's behavior as described in the Verilog code with its formal specifications is essential. Any discrepancies, especially under rare or boundary conditions, could point towards the presence of a Trojan. Formal verification techniques can help prove or disprove that the circuit adheres to its security properties
.
●
Untestable Stuck-at Faults: While not directly related to hardware Trojans, untestable stuck-at faults can complicate Trojan detection
. ATPG techniques can help identify these faults, and removing the corresponding gates from the circuit can simplify the analysis and potentially make Trojan detection easier
.
●
Timing Anomalies: Hardware Trojans might introduce unexpected delays or timing variations in the circuit
. Analyzing the timing characteristics of the circuit as described in the Verilog code, especially under different operating conditions, is crucial. Any significant deviations from the expected timing behavior could indicate the presence of a Trojan. Techniques like path delay fingerprinting can be used to compare timing characteristics with a golden reference
.
●
Suspicious Signals: Certain signals within the circuit might exhibit unusual behavior or patterns that could indicate malicious activity
. These could include signals with unusually low or high switching probabilities, signals that are highly correlated with specific inputs, or signals that appear to be used for unintended purposes. Techniques like signal correlation analysis and clustering can help identify suspicious signals. Equivalence checking can be used to compare these signals against the specifications from the IP provider.""" },
      { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not: " + verilog + """.
    Don't explain anything just answer with 1 if it probably has trojan or 0 otherwise.
    Do not deviate from the 0 or 1 answer pattern."""}
      ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_checklist(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_checklist(design, "meta-llama/Llama-3.3-70B-Instruct", True)

"Evaluating the given Verilog design for potential hardware Trojans requires a thorough analysis of the code, focusing on the aspects mentioned earlier, such as rarely triggered logic, unexecuted code, redundant or unexpected logic, deviations from specifications, untestable stuck-at faults, timing anomalies, and suspicious signals.\n\n### Rarely Triggered Logic\n\nThe design includes several conditional statements and case statements that could potentially lead to rarely triggered logic. For example, the `case(modei)` statement inside the `always @(*)` block has multiple branches that are only executed under specific conditions. However, without knowing the exact input sequences and operating conditions, it's challenging to determine if any of these branches are rarely triggered.\n\n### Unexecuted Code\n\nThe code appears to be well-structured, and most of the variables and signals are used throughout the design. However, there are some signals, such as `barrett_readyi`, `barrett_vali

In [ ]:
detect_trojan_checklist(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_checklist(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

"Evaluating the given Verilog design for potential hardware Trojans involves a thorough analysis of its structure, behavior, and any anomalies that could indicate malicious modifications. Here's a step-by-step evaluation based on the provided aspects to consider:\n\n### Rarely Triggered Logic\n\n1. **Conditional Statements and Rare Conditions**: The design includes several conditional statements (e.g., `if (mode == INVERSE_NTT_MODE)`, `case(modei)`), which could potentially harbor rarely triggered logic. For instance, the condition `if (aj3[4][0] == 1)` within the `INVERSE_NTT_MODE` case might be considered rare, depending on the input data patterns.\n   \n2. **Sequential Logic**: The use of sequential logic (e.g., `always @(posedge clk)`) and shift registers (e.g., `valid_sr`) could also hide rarely triggered conditions, especially if certain states are reached only under specific input sequences.\n\n### Unexecuted Code\n\n1. **Code Coverage**: Without explicit simulation or testing d

In [ ]:
detect_trojan_checklist(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_checklist(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

"Evaluating the given Verilog design for potential hardware Trojans involves a thorough analysis of its structure, behavior, and any anomalies that could indicate malicious modifications. Here's a step-by-step evaluation based on the provided aspects to consider:\n\n### Rarely Triggered Logic\n\n1. **Conditional Statements and Rare Conditions**: The design includes several conditional statements (e.g., `if (mode == INVERSE_NTT_MODE)`, `case(modei)`). These could potentially harbor rarely triggered logic, especially if certain modes or conditions are rarely met during normal operation.\n2. **Specific Input Combinations**: The use of specific input combinations (e.g., `aj5 > DILITHIUM_Q && mode == FORWARD_NTT_MODE`) could be indicative of trigger conditions for potential Trojans.\n\n### Unexecuted Code\n\n1. **Code Coverage**: Without explicit simulation or testing data, it's challenging to determine if there are sections of the code that are never executed. However, the presence of mult

#### 2.4. Perspective-based Analysis - Verification Engineer

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_verification_engineer(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
    { "role": "system", "content": """You are a Verification Engineer. Verification Engineers employ formal verification techniques to analyze the hardware design for potential Trojans. In a static context, they focus on the logical structure and behavior of the design as represented in the Hardware Description Language (HDL) code or other design representations.
Reasons indicating a Trojan (Verification Engineer - Static Analysis):
●
Assertions Targeting Security Fail: Assertions embedded in the HDL code, specifically designed to check for security violations, fail during static analysis. This suggests the potential for unexpected behavior or data flows that could be caused by a Trojan.
●
Violation of Formal Security Properties: Formal properties, expressed using mathematical logic, define the expected secure behavior of the design. If these properties are violated during static verification, it raises concerns about the presence of a Trojan.
●
Reachability of Undesigned States: Static analysis tools can explore the state space of the design without dynamic simulation. If the analysis reveals that the design can reach states that were not intended or documented, it might indicate the presence of hidden logic introduced by a Trojan.
●
Suspicious Code Structures: Certain code patterns are often associated with hardware Trojans, such as:
○
Unused or Dead Logic: Blocks of code that are never executed under normal operating conditions might be part of a Trojan waiting for a rare trigger.
○
Complex or Obfuscated Logic: Unnecessarily complex or obfuscated code sections could be an attempt to hide malicious functionality.
●
Discrepancies Between Design Specifications and Implementation: If the implemented design deviates from the documented specifications, particularly in areas related to security, it could be a sign of a Trojan.
Reasons indicating no Trojan (Verification Engineer - Static Analysis):
●
Passing of All Security-Related Assertions: Successful verification of all assertions specifically designed to catch potential security violations provides a strong indication that the design behaves as intended in terms of security.
●
Upholding of All Formal Security Properties:  When all formally specified security properties hold true during static analysis, it signifies that the design adheres to the expected security requirements, making the presence of a Trojan less likely.
●
Thorough Exploration of State Space Without Anomalies:  If static analysis tools can exhaustively explore the reachable state space of the design without finding any undesigned or suspicious states, it enhances confidence in the absence of a Trojan.
●
Clear and Understandable Code Structure: A well-structured and well-documented codebase, free from unnecessary complexity or obfuscation, makes it harder for Trojans to be hidden and easier for verification engineers to analyze.
●
Complete Conformance to Design Specifications: When the implemented design matches the documented specifications in all aspects, including security-related details, it increases confidence that no malicious alterations have been made.""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not, and justify: " + verilog}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
    { "role": "system", "content": """You are a Verification Engineer. Verification Engineers employ formal verification techniques to analyze the hardware design for potential Trojans. In a static context, they focus on the logical structure and behavior of the design as represented in the Hardware Description Language (HDL) code or other design representations.
Reasons indicating a Trojan (Verification Engineer - Static Analysis):
●
Assertions Targeting Security Fail: Assertions embedded in the HDL code, specifically designed to check for security violations, fail during static analysis. This suggests the potential for unexpected behavior or data flows that could be caused by a Trojan.
●
Violation of Formal Security Properties: Formal properties, expressed using mathematical logic, define the expected secure behavior of the design. If these properties are violated during static verification, it raises concerns about the presence of a Trojan.
●
Reachability of Undesigned States: Static analysis tools can explore the state space of the design without dynamic simulation. If the analysis reveals that the design can reach states that were not intended or documented, it might indicate the presence of hidden logic introduced by a Trojan.
●
Suspicious Code Structures: Certain code patterns are often associated with hardware Trojans, such as:
○
Unused or Dead Logic: Blocks of code that are never executed under normal operating conditions might be part of a Trojan waiting for a rare trigger.
○
Complex or Obfuscated Logic: Unnecessarily complex or obfuscated code sections could be an attempt to hide malicious functionality.
●
Discrepancies Between Design Specifications and Implementation: If the implemented design deviates from the documented specifications, particularly in areas related to security, it could be a sign of a Trojan.
Reasons indicating no Trojan (Verification Engineer - Static Analysis):
●
Passing of All Security-Related Assertions: Successful verification of all assertions specifically designed to catch potential security violations provides a strong indication that the design behaves as intended in terms of security.
●
Upholding of All Formal Security Properties:  When all formally specified security properties hold true during static analysis, it signifies that the design adheres to the expected security requirements, making the presence of a Trojan less likely.
●
Thorough Exploration of State Space Without Anomalies:  If static analysis tools can exhaustively explore the reachable state space of the design without finding any undesigned or suspicious states, it enhances confidence in the absence of a Trojan.
●
Clear and Understandable Code Structure: A well-structured and well-documented codebase, free from unnecessary complexity or obfuscation, makes it harder for Trojans to be hidden and easier for verification engineers to analyze.
●
Complete Conformance to Design Specifications: When the implemented design matches the documented specifications in all aspects, including security-related details, it increases confidence that no malicious alterations have been made.""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not: " + verilog + """.
    Don't explain anything just answer with 1 if it probably has trojan or 0 otherwise.
    Do not deviate from the 0 or 1 answer pattern."""}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_verification_engineer(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_verification_engineer(design, "meta-llama/Llama-3.3-70B-Instruct", True)

"Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my answer.\n\n**Initial Observations**\n\n1. The design is a butterfly module, which is a part of a larger cryptographic system, specifically the CRYSTALS-Dilithium algorithm.\n2. The design has multiple inputs, outputs, and internal signals, which makes it complex and challenging to analyze.\n3. The design uses a Barrett reducer module, which is a common component in cryptographic systems.\n\n**Static Analysis**\n\nTo evaluate the presence of a Trojan, I will perform a static analysis of the design. I will look for suspicious code structures, unused or dead logic, and discrepancies between the design specifications and implementation.\n\n1. **Unused or Dead Logic**: After reviewing the code, I did not find any obvious unused or dead logic. All signals and variables seem to be used in the design.\n2. **Suspicious Code Structures**: The design uses a combination of always blocks, initial blocks

In [ ]:
detect_trojan_verification_engineer(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_verification_engineer(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

'To evaluate if the given Verilog design has a Trojan or not, we need to analyze the code for potential security threats. Based on the provided code, here are some observations:\n\n1. **Unused or Dead Logic**: The code has several variables and signals that are not used or are assigned a constant value. For example, `aj3[0]`, `aj3[1]`, `aj3[2]`, and `aj3[3]` are initialized to 0 but not used anywhere in the code. Similarly, `barrett_readyi` is not used. This could be a potential indication of a Trojan, but it\'s also possible that these signals are leftovers from previous design iterations or are intended for future use.\n\n2. **Complex or Obfuscated Logic**: The code has some complex logic, such as the `case` statement inside the `always @(*)` block, which could be an attempt to hide malicious functionality. However, this complexity could also be due to the design\'s requirements.\n\n3. **Discrepancies Between Design Specifications and Implementation**: Without the design specificatio

In [ ]:
detect_trojan_verification_engineer(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_verification_engineer(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

'Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my answer.\n\n**Initial Observations**\n\n1. The design is a butterfly module, which is a common component in cryptographic algorithms, such as the NTT (Number Theoretic Transform).\n2. The design has multiple modes of operation, including forward NTT, inverse NTT, multiplication, addition, and subtraction.\n3. The design uses a Barrett reducer module, which is a common component in cryptographic algorithms.\n\n**Potential Trojan Indicators**\n\n1. **Unused or Dead Logic**: The design has several registers and wires that are not used in all modes of operation. For example, `ajlen2_FNTT` and `ajlen2_INTT` are only used in specific modes. This could be a potential indicator of a Trojan, as unused logic can be used to hide malicious functionality.\n2. **Complex or Obfuscated Logic**: The design has some complex logic, such as the `sub_tmp` and `subtractor` calculations, which could be used to hid

#### 2.5. Perspective-based Analysis - Security Analyst

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_security_analyst(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
    { "role": "system", "content": """You are a Security Analyst. Security Analysts leverage their knowledge of attack strategies, threat models, and hardware vulnerabilities to scrutinize the design for potential Trojan insertion points and malicious modifications.
Reasons indicating a Trojan (Security Analyst - Static Analysis):
●
Identification of Potential Trigger Mechanisms: Analyzing the design for conditions or signals that could activate a Trojan. This might involve looking for:
○
Rare Event Detectors: Logic that responds to infrequently occurring events might be used to trigger a Trojan's malicious behavior.
○
External Interfaces: Analyzing external interfaces for potential misuse or vulnerabilities that could be exploited to activate a Trojan.
○
Unusual Timing or Clocking Logic:  Trojans might rely on specific timing relationships or clock signals to activate their payload.
●
Detection of Suspicious Data Flows: Tracing data paths within the design to identify:
○
Unauthorized Access to Secure Data:  Trojans may attempt to access or leak confidential information by creating hidden data paths.
○
Bypassing of Security Mechanisms:  Looking for ways a Trojan could circumvent security protocols or access control measures.
●
Analysis of Unused Resources and Functionality: Examining:
○
Unused Memory or Registers:  These could be used to store malicious code or data.
○
Disabled or Hidden Functionality: Analyzing whether these have been tampered with or could be activated maliciously.
●
Scrutiny of Third-Party IP Cores:  If the design incorporates IP cores from external sources, a security analyst would pay particular attention to these, as they represent potential points of Trojan insertion.
●
Review of Design for Trustworthiness (DfT) Measures:  Evaluating whether appropriate DfT techniques have been incorporated into the design to make it more resilient to Trojan attacks. If DfT measures are weak or absent, it increases the risk of Trojans.
Reasons indicating no Trojan (Security Analyst - Static Analysis):
●
Adherence to Secure Design Principles: A design that demonstrably follows established secure design principles, such as secure coding guidelines, least privilege access, and compartmentalization, is less likely to harbor Trojans.
●
Absence of Known Trojan Patterns:  Not finding any code structures or design elements commonly associated with hardware Trojans (e.g., unused logic, rare triggers, suspicious data flows) provides a positive indicator.
●
Strong Security Verification Results: If the design has undergone rigorous security verification, including formal methods and manual code review by security experts, and no vulnerabilities have been identified, it enhances confidence in the absence of Trojans.
●
Utilization of Trusted Design Sources: Employing only trusted third-party IP cores and design tools from reputable vendors significantly reduces the risk of Trojan insertion.
●
Incorporation of Robust Design for Trust (DfT) Features:  A design that incorporates strong DfT measures, like logic obfuscation, tamper detection mechanisms, or runtime monitoring, demonstrates a proactive approach to security and makes it much harder for Trojans to be successfully inserted or activated.
""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not, and justify: " + verilog}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
    { "role": "system", "content": """You are a Security Analyst. Security Analysts leverage their knowledge of attack strategies, threat models, and hardware vulnerabilities to scrutinize the design for potential Trojan insertion points and malicious modifications.
Reasons indicating a Trojan (Security Analyst - Static Analysis):
●
Identification of Potential Trigger Mechanisms: Analyzing the design for conditions or signals that could activate a Trojan. This might involve looking for:
○
Rare Event Detectors: Logic that responds to infrequently occurring events might be used to trigger a Trojan's malicious behavior.
○
External Interfaces: Analyzing external interfaces for potential misuse or vulnerabilities that could be exploited to activate a Trojan.
○
Unusual Timing or Clocking Logic:  Trojans might rely on specific timing relationships or clock signals to activate their payload.
●
Detection of Suspicious Data Flows: Tracing data paths within the design to identify:
○
Unauthorized Access to Secure Data:  Trojans may attempt to access or leak confidential information by creating hidden data paths.
○
Bypassing of Security Mechanisms:  Looking for ways a Trojan could circumvent security protocols or access control measures.
●
Analysis of Unused Resources and Functionality: Examining:
○
Unused Memory or Registers:  These could be used to store malicious code or data.
○
Disabled or Hidden Functionality: Analyzing whether these have been tampered with or could be activated maliciously.
●
Scrutiny of Third-Party IP Cores:  If the design incorporates IP cores from external sources, a security analyst would pay particular attention to these, as they represent potential points of Trojan insertion.
●
Review of Design for Trustworthiness (DfT) Measures:  Evaluating whether appropriate DfT techniques have been incorporated into the design to make it more resilient to Trojan attacks. If DfT measures are weak or absent, it increases the risk of Trojans.
Reasons indicating no Trojan (Security Analyst - Static Analysis):
●
Adherence to Secure Design Principles: A design that demonstrably follows established secure design principles, such as secure coding guidelines, least privilege access, and compartmentalization, is less likely to harbor Trojans.
●
Absence of Known Trojan Patterns:  Not finding any code structures or design elements commonly associated with hardware Trojans (e.g., unused logic, rare triggers, suspicious data flows) provides a positive indicator.
●
Strong Security Verification Results: If the design has undergone rigorous security verification, including formal methods and manual code review by security experts, and no vulnerabilities have been identified, it enhances confidence in the absence of Trojans.
●
Utilization of Trusted Design Sources: Employing only trusted third-party IP cores and design tools from reputable vendors significantly reduces the risk of Trojan insertion.
●
Incorporation of Robust Design for Trust (DfT) Features:  A design that incorporates strong DfT measures, like logic obfuscation, tamper detection mechanisms, or runtime monitoring, demonstrates a proactive approach to security and makes it much harder for Trojans to be successfully inserted or activated.
""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not: " + verilog + """.
    Don't explain anything just answer with 1 if it probably has trojan or 0 otherwise.
    Do not deviate from the 0 or 1 answer pattern."""}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_security_analyst(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_security_analyst(design, "meta-llama/Llama-3.3-70B-Instruct", True)

"Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my findings.\n\n**Initial Observations**\n\n1. The design appears to be a cryptographic implementation, specifically a butterfly module for the CRYSTALS-Dilithium algorithm.\n2. The code is well-structured, and the comments suggest that it is based on a research paper.\n3. The design uses a mix of combinational and sequential logic, with a focus on arithmetic operations.\n\n**Static Analysis**\n\nTo detect potential Trojans, I will apply the following checks:\n\n1. **Identification of Potential Trigger Mechanisms**:\n\t* I did not find any obvious rare event detectors or unusual timing/clocking logic that could trigger a Trojan.\n\t* The design uses a standard clock signal (`clk`) and a reset signal (`rst`).\n2. **Detection of Suspicious Data Flows**:\n\t* I did not find any unauthorized access to secure data or bypassing of security mechanisms.\n\t* The design uses a Barrett reducer module, w

In [ ]:
detect_trojan_security_analyst(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_security_analyst(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

'Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my findings.\n\n**Initial Observations**\n\n1. The design appears to be a cryptographic module, specifically a butterfly module, which is a common component in cryptographic algorithms.\n2. The design uses a mix of combinational and sequential logic, with several registers and wires.\n3. The design has multiple modes of operation, including forward and inverse NTT (Number Theoretic Transform) modes, multiplication mode, addition mode, and subtraction mode.\n\n**Potential Trojan Indicators**\n\n1. **Unused Resources and Functionality**: The design has several unused registers and wires, such as `aj3[0]`, `aj3[1]`, `aj3[2]`, and `aj3[3]`, which could potentially be used to store malicious code or data.\n2. **Suspicious Data Flows**: The design has several data flows that could be considered suspicious, such as the use of `DILITHIUM_Q` in multiple places, which could potentially be used to leak c

In [ ]:
detect_trojan_security_analyst(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_security_analyst(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

"Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my findings.\n\n**Initial Observations**\n\n1. The design appears to be a cryptographic module, specifically a butterfly module, which is a common component in cryptographic algorithms.\n2. The design uses a combination of arithmetic and logical operations, which is typical for cryptographic modules.\n3. The design has multiple modes of operation, including forward and inverse NTT (Number Theoretic Transform) modes, multiplication mode, addition mode, and subtraction mode.\n\n**Potential Trojan Indicators**\n\n1. **Unused Resources and Functionality**: The design has several unused registers and wires, such as `aj3[0]`, `aj3[1]`, `aj3[2]`, and `aj3[3]`, which could potentially be used to store malicious code or data.\n2. **Suspicious Data Flows**: The design has several data flows that could be considered suspicious, such as the use of `DILITHIUM_Q` in multiple places, which could potentially 

#### 2.6. Perspective-based Analysis - Test Engineer

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=token)

def detect_trojan_test_engineer(verilog, model_id, verbose):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  if verbose:
    messages = [
    { "role": "system", "content": """You are a Test Engineer. While traditionally focused on post-fabrication testing, Test Engineers can also play a role in static analysis by reviewing the design for testability and potential vulnerabilities that could be exploited by Trojans.
Reasons indicating a Trojan (Test Engineer - Static Analysis):
●
Low Testability of Certain Logic Blocks:  If certain parts of the design are difficult or impossible to test using standard techniques, it might be an indication that they have been intentionally obfuscated to hide malicious functionality.
●
Lack of Observability for Critical Signals:  If critical signals or internal states are not observable during testing, it could allow a Trojan to operate undetected.
●
Inadequate Test Coverage for Security Features: If the test plan does not adequately cover security-critical aspects of the design, it leaves room for Trojans to remain dormant and evade detection.
●
Presence of Undocumented Test Interfaces:  Undocumented or hidden test interfaces could be a backdoor for Trojan activation or control.
●
Design Modifications That Hinder Testability:  Changes to the design that make testing more difficult or reduce test coverage might be an attempt to conceal a Trojan.
Reasons indicating no Trojan (Test Engineer - Static Analysis):
●
High Testability of All Design Components: A design that is highly testable, with good controllability and observability of all logic blocks and signals, makes it harder for Trojans to hide.
●
Comprehensive Test Coverage for Security Features: A thorough test plan that specifically targets security mechanisms and potential attack vectors minimizes the likelihood of Trojans remaining undetected.
●
Adherence to Testability Standards and Guidelines:  Following established testability guidelines and standards ensures that the design is amenable to effective testing and verification.
●
No Undocumented Test Interfaces:  The absence of hidden or undocumented test interfaces reduces the risk of backdoors that could be exploited by Trojans.
●
Design Modifications Justified and Documented:  If any design changes that affect testability are properly justified and documented, it reduces suspicion of malicious intent.""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not, and justify: " + verilog}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=5000,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    return response_content

  else:
    messages = [
    { "role": "system", "content": """You are a Test Engineer. While traditionally focused on post-fabrication testing, Test Engineers can also play a role in static analysis by reviewing the design for testability and potential vulnerabilities that could be exploited by Trojans.
Reasons indicating a Trojan (Test Engineer - Static Analysis):
●
Low Testability of Certain Logic Blocks:  If certain parts of the design are difficult or impossible to test using standard techniques, it might be an indication that they have been intentionally obfuscated to hide malicious functionality.
●
Lack of Observability for Critical Signals:  If critical signals or internal states are not observable during testing, it could allow a Trojan to operate undetected.
●
Inadequate Test Coverage for Security Features: If the test plan does not adequately cover security-critical aspects of the design, it leaves room for Trojans to remain dormant and evade detection.
●
Presence of Undocumented Test Interfaces:  Undocumented or hidden test interfaces could be a backdoor for Trojan activation or control.
●
Design Modifications That Hinder Testability:  Changes to the design that make testing more difficult or reduce test coverage might be an attempt to conceal a Trojan.
Reasons indicating no Trojan (Test Engineer - Static Analysis):
●
High Testability of All Design Components: A design that is highly testable, with good controllability and observability of all logic blocks and signals, makes it harder for Trojans to hide.
●
Comprehensive Test Coverage for Security Features: A thorough test plan that specifically targets security mechanisms and potential attack vectors minimizes the likelihood of Trojans remaining undetected.
●
Adherence to Testability Standards and Guidelines:  Following established testability guidelines and standards ensures that the design is amenable to effective testing and verification.
●
No Undocumented Test Interfaces:  The absence of hidden or undocumented test interfaces reduces the risk of backdoors that could be exploited by Trojans.
●
Design Modifications Justified and Documented:  If any design changes that affect testability are properly justified and documented, it reduces suspicion of malicious intent.""" },
    { "role": "user", "content": "Given a Verilog design, evaluate if it has a trojan or not: " + verilog + """.
    Don't explain anything just answer with 1 if it probably has trojan or 0 otherwise.
    Do not deviate from the 0 or 1 answer pattern."""}
  ]

    output = client.chat.completions.create(
      model=model_id,
      messages=messages,
      temperature=0.5,
      max_tokens=10,
      top_p=0.5
    )

    # Extract the content of the model's response
    response_content = output.choices[0].message['content']
    #print(response_content)

    # Remove any non-digit characters from the response content
    response_content = ''.join(filter(str.isdigit, response_content))
    #print(response_content)

    # Return the integer value or 0 if response is empty
    return int(response_content[0]) if response_content else 0

In [ ]:
detect_trojan_test_engineer(design, "meta-llama/Llama-3.3-70B-Instruct", False)

0

In [ ]:
detect_trojan_test_engineer(design, "meta-llama/Llama-3.3-70B-Instruct", True)

'Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my answer.\n\n**Initial Observations**\n\n1. The design is a butterfly module, which is a part of a larger cryptographic system (CRYSTALS-Dilithium).\n2. The design has a complex structure with multiple modes of operation (FORWARD_NTT_MODE, INVERSE_NTT_MODE, MULT_MODE, ADD_MODE, and SUB_MODE).\n3. The design uses a Barrett reducer module, which is a common component in cryptographic systems.\n\n**Reasons indicating no Trojan**\n\n1. **High Testability of All Design Components**: The design has a clear and well-structured architecture, making it easier to test and verify.\n2. **Comprehensive Test Coverage for Security Features**: The design includes a comprehensive set of test signals and modes, which suggests that the designers have considered security testing and verification.\n3. **Adherence to Testability Standards and Guidelines**: The design follows standard Verilog coding practices and i

In [ ]:
detect_trojan_test_engineer(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_test_engineer(design_no_comments, "meta-llama/Llama-3.3-70B-Instruct", True)

"Based on the provided Verilog design, I will evaluate if it has a Trojan or not, and justify my conclusion.\n\n**Initial Observations**\n\n1. The design is a butterfly module, which is a common component in cryptographic and signal processing applications.\n2. The module has multiple modes of operation, including forward and inverse NTT (Number Theoretic Transform), multiplication, addition, and subtraction.\n3. The design uses a Barrett reducer module, which is a common technique for reducing the size of integers in modular arithmetic.\n\n**Reasons indicating no Trojan**\n\n1. **High Testability of All Design Components**: The design has a clear and structured architecture, with well-defined inputs, outputs, and internal signals. This makes it easier to test and verify the functionality of each component.\n2. **Comprehensive Test Coverage for Security Features**: The design includes multiple modes of operation, which can be tested individually to ensure that the security features are

In [ ]:
detect_trojan_test_engineer(trojan, "meta-llama/Llama-3.3-70B-Instruct", False)

1

In [ ]:
detect_trojan_test_engineer(trojan, "meta-llama/Llama-3.3-70B-Instruct", True)

"Evaluating the given Verilog design for potential Trojans involves analyzing its testability, observability, and adherence to testability standards. Based on the provided code, here are some observations:\n\n### Reasons indicating a potential Trojan:\n\n1. **Low Testability of Certain Logic Blocks**: The design contains complex logic blocks, such as the `Barrett` module, which might be difficult to test thoroughly. The presence of many internal signals and variables (e.g., `aj3`, `ajlen2_FNTT`, `ajlen2_INTT`, `ajlen2_MODM`) could make it challenging to achieve high test coverage.\n2. **Lack of Observability for Critical Signals**: Some critical signals, like `aj3`, `ajlen3`, and `ajlen5`, are not directly observable from the module's outputs. This lack of observability could make it harder to detect malicious behavior.\n3. **Inadequate Test Coverage for Security Features**: The design appears to implement cryptographic functions (e.g., NTT, multiplication, and addition in a finite fie

#### 2.7. Simple Prompt - DeepSeek R1

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/novita/v3/openai",
    api_key=token,
)

def detect_trojan_deepseek(verilog):

  # Truncate the Verilog code to a maximum number of tokens
  max_input_tokens = 5000  # Adjust this value as needed
  verilog = verilog[:max_input_tokens]

  completion = client.chat.completions.create(
      model="deepseek/deepseek-r1-turbo",
      messages = [
        { "role": "system", "content": "You are a expert in hardware security and has extensive knowledge regarding hardware trojans in Verilog designs." },
        { "role": "user", "content": "Evaluate if the following Verilog design has a trojan or not, and justify: " + verilog }
        ],
      max_tokens=512,
  )

  return completion.choices[0].message.content

In [ ]:
detect_trojan_deepseek(design)

'<think>\nOkay, let\'s take a look at this Verilog code and try to figure out if there\'s a hardware Trojan in here. The module is called "butterfly" which makes me think it\'s part of a Number Theoretic Transform (NTT) used in cryptographic algorithms like CRYSTALS-Dilithium. The code has several modes, like forward and inverse NTT, multiply, add, subtract, etc. \n\nFirst, I\'ll check for any suspicious signals or logic that doesn\'t align with the intended functionality. Hardware Trojans often have triggers and payloads. Triggers could be rare signals or specific conditions, while payloads could be leaking data, causing faults, or bypassing security.\n\nLooking at the inputs and outputs: there\'s a clock, reset, mode, validi, aj, ajlen, zeta, and outputs bj, bjlen, valido. The parameters include DILITHIUM_Q, which is a modulus value. The module uses a Barrett reducer, which is for modular reduction. \n\nIn the always block with @(*), there\'s combinational logic assigning various sig

In [ ]:
detect_trojan_deepseek(design_no_comments)

'<think>\nOkay, let\'s try to figure out if this Verilog design has a hardware Trojan. I\'m not an expert, but I\'ll go through the code step by step.\n\nFirst, the module is called "butterfly," which makes me think it\'s part of a Fast Fourier Transform (FFT) or Number Theoretic Transform (NTT) used in cryptography like Dilithium. The inputs include a clock, reset, mode, validi, aj, ajlen, zeta, and outputs bj, bjlen, valido. The parameters include DILITHIUM_Q and various modes like forward and inverse NTT, multiply, add, subtract.\n\nLooking at the registers declared: there\'s a bunch of aj and ajlen variables with different suffixes (aj1, aj2, etc.), which might be pipeline stages. There\'s also a Barrett module instantiated, which is probably for modular reduction. The initial block initializes valid_sr and aj3 array to zero, which seems normal.\n\nIn the always @(*) block, they handle combinational logic for different modes. For example, in FORWARD_NTT_MODE, they set adda, addb, s

In [ ]:
detect_trojan_deepseek(trojan)

'<think>\nOkay, let\'s try to figure out if this Verilog code has a hardware Trojan. First, I need to understand what the module is supposed to do. The module is called "butterfly," which might relate to a Fast Fourier Transform (FFT) butterfly operation, commonly used in signal processing. The inputs include clock, reset, mode, validi, aj, ajlen, zeta, and outputs are bj, bjlen, valido. There are some parameters like DILITHIUM_Q and different modes (FORWARD_NTT_MODE, INVERSE_NTT_MODE, etc.), which suggests it\'s part of a Number Theoretic Transform (NTT) used in lattice-based cryptography, maybe related to the Dilithium algorithm.\n\nNow, looking for potential Trojans. A hardware Trojan is a malicious modification to the circuit that can cause unintended behavior, like leaking secrets, causing errors, or creating backdoors. Red flags might be unexpected assignments, hidden state transitions, or data leaks.\n\nLooking at the code, there\'s a register called "leak" which is assigned in 